# 제품 이상여부 판별 프로젝트


## 데이터 불러오기


### 필수 라이브러리


In [1]:
import os
from pprint import pprint

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import random
import torch
import warnings
warnings.filterwarnings("ignore")

### 데이터 읽어오기


In [2]:
ROOT_DIR = "data"
RANDOM_STATE = 736665
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(RANDOM_STATE)


seed_everything(RANDOM_STATE)
# Load data
train_data = pd.read_csv(os.path.join(ROOT_DIR, "train.csv"))
test_data = pd.read_csv(os.path.join(ROOT_DIR, "test.csv"))

In [3]:
# test_data에는 target 컬럼이 있지만 다 결측치
# test_data에 Set ID가 있어서 열 하나가 train_data보다 많음
train_data.shape, test_data.shape

((40506, 464), (17361, 465))

## 데이터 전처리

### 전체 행이 결측치이거나 똑같은 값인 칼럼 삭제

In [4]:
import pandas as pd

def drop_columns(df):

    drop_cols = []
    # 전체 행이 결측치인 칼럼 삭제
    base_col = df.columns
    df.dropna(axis=1, how='all', inplace=True)
    # drop된 컬럼 확인
    dif_col = list(set(base_col) - set(df.columns))
    drop_cols.extend(dif_col)
    print('모든 값이 결측치인 열 제거 진행')
    print(f'drop한 컬럼 : {dif_col}')
    print(f'drop한 컬럼 개수 : {len(dif_col)}개')

    # 특정 컬럼의 값이 다 같은 경우 컬럼 제거
    base_col = df.columns
    for col in df.columns:
        if (df[col].nunique() == 1) and (df[col].isnull().sum() == 0):
            df.drop(columns=col, inplace=True)
    dif_col = list(set(base_col) - set(df.columns))
    drop_cols.extend(dif_col)
    print('모든 값이 같은 열 제거 진행')
    print(f'drop한 컬럼 : {dif_col}')
    print(f'drop한 컬럼 개수 : {len(dif_col)}개')


    return df, drop_cols

train_data, drop_cols = drop_columns(train_data)
test_data = test_data.drop(columns=drop_cols)
train_data.shape, test_data.shape

모든 값이 결측치인 열 제거 진행
drop한 컬럼 : ['Dispense Volume(Stage1) Judge Value_Fill1', 'Dispense Volume(Stage2) Unit Time_Fill2', 'HEAD Standby Position X Judge Value_Fill2', 'Stage2 Circle2 Distance Speed Judge Value_Dam', 'Stage2 Circle4 Distance Speed Judge Value_Dam', 'Head Purge Position Z Judge Value_Fill1', 'Stage1 Line4 Distance Speed Unit Time_Dam', 'CURE STANDBY POSITION Θ Judge Value_Dam', 'Stage3 Circle2 Distance Speed Judge Value_Dam', 'HEAD Standby Position Z Unit Time_Dam', 'HEAD NORMAL COORDINATE Y AXIS(Stage1) Judge Value_Fill2', 'Head Purge Position X Unit Time_Dam', 'HEAD NORMAL COORDINATE Z AXIS(Stage1) Unit Time_Fill1', 'DISCHARGED TIME OF RESIN(Stage3) Judge Value_Fill2', 'CURE SPEED Judge Value_Dam', 'CURE START POSITION Θ Unit Time_Fill2', 'Stage1 Circle1 Distance Speed Judge Value_Dam', 'CURE END POSITION X Judge Value_Fill2', 'THICKNESS 1 Judge Value_Dam', 'Stage1 Circle3 Distance Speed Judge Value_Dam', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Unit Time_Dam', 'Stage1 Line

((40506, 151), (17361, 152))

### 중복 컬럼 제거

In [5]:
columns_dict = {}
duplicate_dict = {}

for col in train_data.columns:
    # 각 열의 해시 값을 계산하여 문자열로 변환
    col_hash = tuple(train_data[col].tolist())

    # 이미 존재하는 해시 값이라면 중복으로 처리
    if col_hash in columns_dict:
        existing_col = columns_dict[col_hash]
        if existing_col in duplicate_dict:
            duplicate_dict[existing_col].append(col)
        else:
            duplicate_dict[existing_col] = [col]
    else:
        columns_dict[col_hash] = col
duplicate_dict

{'CURE END POSITION Θ Collect Result_Dam': ['CURE START POSITION Θ Collect Result_Dam'],
 'HEAD NORMAL COORDINATE Z AXIS(Stage2) Collect Result_Dam': ['HEAD NORMAL COORDINATE Z AXIS(Stage3) Collect Result_Dam'],
 'HEAD Standby Position Y Collect Result_Dam': ['Head Purge Position Y Collect Result_Dam'],
 'Stage1 Circle2 Distance Speed Collect Result_Dam': ['Stage1 Circle3 Distance Speed Collect Result_Dam',
  'Stage1 Circle4 Distance Speed Collect Result_Dam'],
 'Stage1 Line1 Distance Speed Collect Result_Dam': ['Stage1 Line3 Distance Speed Collect Result_Dam'],
 'Stage2 Circle2 Distance Speed Collect Result_Dam': ['Stage2 Circle3 Distance Speed Collect Result_Dam',
  'Stage2 Circle4 Distance Speed Collect Result_Dam',
  'Stage2 Line1 Distance Speed Collect Result_Dam'],
 'Stage3 Circle2 Distance Speed Collect Result_Dam': ['Stage3 Circle3 Distance Speed Collect Result_Dam',
  'Stage3 Circle4 Distance Speed Collect Result_Dam'],
 'Stage3 Line1 Distance Speed Collect Result_Dam': ['Stag

In [6]:
delete_cols = []
for col in list(duplicate_dict.values()):
    delete_cols.extend(col)
delete_cols

['CURE START POSITION Θ Collect Result_Dam',
 'HEAD NORMAL COORDINATE Z AXIS(Stage3) Collect Result_Dam',
 'Head Purge Position Y Collect Result_Dam',
 'Stage1 Circle3 Distance Speed Collect Result_Dam',
 'Stage1 Circle4 Distance Speed Collect Result_Dam',
 'Stage1 Line3 Distance Speed Collect Result_Dam',
 'Stage2 Circle3 Distance Speed Collect Result_Dam',
 'Stage2 Circle4 Distance Speed Collect Result_Dam',
 'Stage2 Line1 Distance Speed Collect Result_Dam',
 'Stage3 Circle3 Distance Speed Collect Result_Dam',
 'Stage3 Circle4 Distance Speed Collect Result_Dam',
 'Stage3 Line3 Distance Speed Collect Result_Dam',
 'Model.Suffix_AutoClave',
 'Model.Suffix_Fill1',
 'Model.Suffix_Fill2',
 'Workorder_AutoClave',
 'Workorder_Fill1',
 'Workorder_Fill2',
 'GMES_ORIGIN_INSP_JUDGE_CODE Collect Result_AutoClave',
 'GMES_ORIGIN_INSP_JUDGE_CODE Judge Value_AutoClave',
 'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill1',
 'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill2',
 'Head Pur

In [7]:
train_data.drop(columns=delete_cols, inplace=True)
test_data.drop(columns=delete_cols, inplace=True)
train_data.shape, test_data.shape

((40506, 125), (17361, 126))

### 파생 변수 생성

#### 상관관계가 1, -1인 열 확인 - 할지 말지는 선택
- 특정 열들의 unique값이 아주 적은 경우

In [8]:
# train_data의 수치형 데이터
num_cols = train_data.select_dtypes(include=["int64", "float64"]).columns
corr_matrix = train_data[num_cols].corr()

# 상관계수 1 또는 -1인 쌍만 추출
strong_corrs = corr_matrix[(corr_matrix >= 0.99999) | (corr_matrix <= -0.99999) ]

train_data[['CURE END POSITION X Collect Result_Dam', 'CURE END POSITION Z Collect Result_Dam']].drop_duplicates()

col_list = strong_corrs.columns
corr_list = []
for col in col_list:
    corr_cols = list(strong_corrs[strong_corrs[col].notnull()].index)

    if (len(corr_cols) > 1) and (corr_cols not in corr_list):
        print('서로 상관관계가 1인 집단 크기 :', len(corr_cols))
        corr_list.append(corr_cols)


print('집단 별로 합하기 전의 고유값과 합한 후의 고유값 비교')
for i in range(len(corr_list)):
    print('전', len(train_data[corr_list[i]].drop_duplicates()), '후', train_data[corr_list[i]].sum(axis=1).nunique())

print('집단 별로 합하기 전의 고유값과 합한 후의 고유값 비교')
for i in range(len(corr_list)):
    print('전', len(train_data[corr_list[i]].drop_duplicates()), '후', train_data[corr_list[i]].sum(axis=1).nunique())

# 합하기 전과 후의 고유값이 같은 경우는 그냥 합침
for i in range(4):
    train_data['SUM ' + corr_list[i][0]] = train_data[corr_list[i]].sum(axis=1)
    test_data['SUM ' + corr_list[i][0]] = test_data[corr_list[i]].sum(axis=1)
    print(corr_list[i])
    train_data.drop(columns=corr_list[i], inplace=True)
    test_data.drop(columns=corr_list[i], inplace=True)
train_data.shape, test_data.shape

train_data[corr_list[4]].drop_duplicates()

temp = train_data[corr_list[4]].drop_duplicates()
temp['sum'] =  temp[corr_list[4][0]] + ((temp[corr_list[4][1]] - temp[corr_list[4][0]]) / temp[corr_list[4][0]] * 100)
temp['sum']

train_data['DELTA ' + corr_list[4][0]] = train_data[corr_list[4][0]] + ((train_data[corr_list[4][1]] - train_data[corr_list[4][0]]) / train_data[corr_list[4][0]] * 100)
test_data['DELTA ' + corr_list[4][0]] = test_data[corr_list[4][0]] + ((test_data[corr_list[4][1]] - test_data[corr_list[4][0]]) / test_data[corr_list[4][0]] * 100)
train_data.drop(columns=corr_list[4], inplace=True)
test_data.drop(columns=corr_list[4], inplace=True)
train_data.shape, test_data.shape

train_data[corr_list[5]].drop_duplicates()

train_data['DELTA ' + corr_list[5][0]] = train_data[corr_list[5][0]] - train_data[corr_list[5][1]]
test_data['DELTA ' + corr_list[5][0]] = test_data[corr_list[5][0]] - test_data[corr_list[5][1]]
train_data.drop(columns=corr_list[5], inplace=True)
test_data.drop(columns=corr_list[5], inplace=True)
train_data.shape, test_data.shape

train_data[corr_list[6]].drop_duplicates()

train_data['DELTA ' + corr_list[6][0]]  = (train_data[corr_list[6][1]] - train_data[corr_list[6][0]] + train_data[corr_list[6][2]])
test_data['DELTA ' + corr_list[6][0]]  = (test_data[corr_list[6][1]] - test_data[corr_list[6][0]] + test_data[corr_list[6][2]])
train_data.drop(columns=corr_list[6], inplace=True)
test_data.drop(columns=corr_list[6], inplace=True)
train_data.shape, test_data.shape

서로 상관관계가 1인 집단 크기 : 4
서로 상관관계가 1인 집단 크기 : 18
서로 상관관계가 1인 집단 크기 : 2
서로 상관관계가 1인 집단 크기 : 2
서로 상관관계가 1인 집단 크기 : 2
서로 상관관계가 1인 집단 크기 : 2
서로 상관관계가 1인 집단 크기 : 3
집단 별로 합하기 전의 고유값과 합한 후의 고유값 비교
전 2 후 2
전 2 후 2
전 5 후 5
전 10 후 10
전 26 후 22
전 2 후 1
전 3 후 2
집단 별로 합하기 전의 고유값과 합한 후의 고유값 비교
전 2 후 2
전 2 후 2
전 5 후 5
전 10 후 10
전 26 후 22
전 2 후 1
전 3 후 2
['CURE END POSITION X Collect Result_Dam', 'CURE END POSITION Z Collect Result_Dam', 'CURE END POSITION Θ Collect Result_Dam', 'CURE START POSITION X Collect Result_Dam']
['HEAD Standby Position Y Collect Result_Dam', 'HEAD Standby Position Z Collect Result_Dam', 'Head Clean Position X Collect Result_Dam', 'Head Clean Position Y Collect Result_Dam', 'Head Zero Position X Collect Result_Dam', 'HEAD Standby Position Y Collect Result_Fill1', 'HEAD Standby Position Z Collect Result_Fill1', 'Head Clean Position X Collect Result_Fill1', 'Head Clean Position Y Collect Result_Fill1', 'Head Clean Position Z Collect Result_Fill1', 'Head Purge Position X Collect Res

((40506, 99), (17361, 100))

#### 상관관계 범위를 1, -1이 아닌 0.99, -0.99로 변경 시 확인

In [9]:
num_cols = train_data.select_dtypes(include=["int64", "float64"]).columns
corr_matrix = train_data[num_cols].corr()

# 상관계수 1 또는 -1인 쌍만 추출
strong_corrs = corr_matrix[(corr_matrix >= 0.99)  | (corr_matrix <= -0.99) ]
strong_corrs

# 상관 행렬 계산 (예: df는 데이터프레임)
corr_matrix = train_data[num_cols].corr()

# 절대값이 0.99 이상인 상관관계 쌍을 필터링
high_corr_pairs = corr_matrix[(corr_matrix.abs() >= 0.99)]
high_corr_pairs.fillna(0, inplace=True)
# 상관관계가 0.99 이상인 컬럼 쌍 추출
correlated_pairs = []
for i in range(len(high_corr_pairs.columns)):
    for j in range(i):
        if (high_corr_pairs.iloc[i, j] != 0):
            correlated_pairs.append((high_corr_pairs.index[i], high_corr_pairs.columns[j], high_corr_pairs.iloc[i, j]))

# 결과 출력
for pair in correlated_pairs:
    print(f'{pair[0]}와 {pair[1]}의 상관관계: {pair[2]}')

temp = pd.DataFrame(strong_corrs.notnull().sum())
temp.columns = ['group_size']
temp = temp[temp['group_size'] > 1]
print(temp['group_size'].unique())
temp


col_list = strong_corrs.columns
corr_list = []
for col in col_list:
    corr_cols = list(strong_corrs[strong_corrs[col].notnull()].index)

    if (len(corr_cols) > 1) and (corr_cols not in corr_list):
        print('서로 상관관계가 0.99이상인 집단 크기 :', len(corr_cols))
        corr_list.append(corr_cols)

print('집단 별로 합하기 전의 고유값과 합한 후의 고유값 비교')
for i in range(len(corr_list)):
    print('전', len(train_data[corr_list[i]].drop_duplicates()), '후', train_data[corr_list[i]].sum(axis=1).nunique())
    if len(train_data[corr_list[i]].drop_duplicates()) == train_data[corr_list[i]].sum(axis=1).nunique():
        print(i)

train_data.shape, test_data.shape

# 합하기 전과 후의 고유값이 같은 경우는 그냥 합침
for i in [2, 9]:
    train_data['SUM ' + corr_list[i][0]] = train_data[corr_list[i]].sum(axis=1)
    test_data['SUM ' + corr_list[i][0]] = test_data[corr_list[i]].sum(axis=1)
    print(corr_list[i])
    train_data.drop(columns=corr_list[i], inplace=True)
    test_data.drop(columns=corr_list[i], inplace=True)
train_data.shape, test_data.shape

temp = train_data[corr_list[0]].drop_duplicates()
print(len(temp))
# temp['sum'] =  temp[corr_list[0][0]] + ((temp[corr_list[0][1]] - temp[corr_list[0][0]]) / temp[corr_list[0][0]] * 100)
temp['sum'] =  temp[corr_list[0][0]] + (temp[corr_list[0][1]] - temp[corr_list[0][0]]) / temp[corr_list[0][0]] * 100
temp['sum'].nunique()

corr_list[1]

temp = train_data[corr_list[1]].drop_duplicates()
print(len(temp))
# temp['sum'] =  temp[corr_list[0][0]] + ((temp[corr_list[0][1]] - temp[corr_list[0][0]]) / temp[corr_list[0][0]] * 100)
temp['sum'] =  train_data[corr_list[1][0]] + (train_data[corr_list[1][1]] - train_data[corr_list[1][0]]) / train_data[corr_list[1][0]] * 100
temp['sum'].nunique()

train_data['DELTA ' + corr_list[1][0]] = train_data[corr_list[1][0]] + (train_data[corr_list[1][1]] - train_data[corr_list[1][0]]) / train_data[corr_list[1][0]] * 100
test_data['DELTA ' + corr_list[1][0]] = test_data[corr_list[1][0]] + (test_data[corr_list[1][1]] - test_data[corr_list[1][0]]) / test_data[corr_list[1][0]] * 100
train_data.drop(columns=corr_list[1], inplace=True)
test_data.drop(columns=corr_list[1], inplace=True)
train_data.shape, test_data.shape

corr_list[8]

temp = train_data[corr_list[8]].drop_duplicates()
print(len(temp))
# temp['sum'] =  temp[corr_list[0][0]] + ((temp[corr_list[0][1]] - temp[corr_list[0][0]]) / temp[corr_list[0][0]] * 100)
temp['sum'] = train_data[corr_list[8]].sum(axis=1) + train_data[corr_list[8][2]]
temp['sum'].nunique()

train_data['DELTA ' + corr_list[8][0]] = train_data[corr_list[8]].sum(axis=1) + train_data[corr_list[8][2]]
test_data['DELTA ' + corr_list[8][0]] = test_data[corr_list[8]].sum(axis=1) + test_data[corr_list[8][2]]
train_data.drop(columns=corr_list[8], inplace=True)
test_data.drop(columns=corr_list[8], inplace=True)
train_data.shape, test_data.shape

corr_list[10]

temp = train_data[corr_list[10]].drop_duplicates()
print(len(temp))
# temp['sum'] =  temp[corr_list[0][0]] + ((temp[corr_list[0][1]] - temp[corr_list[0][0]]) / temp[corr_list[0][0]] * 100)
temp['sum'] = (train_data[corr_list[10][5]] + train_data[corr_list[10][6]])/2 + (train_data[corr_list[10][0:5]].sum(axis=1)+ train_data[corr_list[10][7:]].sum(axis=1)) / 7
temp['sum'].nunique()

train_data['DELTA ' + corr_list[10][0]] = (train_data[corr_list[10][5]] + train_data[corr_list[10][6]])/2 + (train_data[corr_list[10][0:5]].sum(axis=1)+ train_data[corr_list[10][7:]].sum(axis=1)) / 7
test_data['DELTA ' + corr_list[10][0]] = (test_data[corr_list[10][5]] + test_data[corr_list[10][6]])/2 + (test_data[corr_list[10][0:5]].sum(axis=1)+ test_data[corr_list[10][7:]].sum(axis=1)) / 7
train_data.drop(columns=corr_list[10], inplace=True)
test_data.drop(columns=corr_list[10], inplace=True)
train_data.shape, test_data.shape

DISCHARGED TIME OF RESIN(Stage3) Collect Result_Dam와 DISCHARGED TIME OF RESIN(Stage1) Collect Result_Dam의 상관관계: 0.9994755091064483
Dispense Volume(Stage3) Collect Result_Dam와 Dispense Volume(Stage1) Collect Result_Dam의 상관관계: 0.9993794627149966
HEAD NORMAL COORDINATE Y AXIS(Stage3) Collect Result_Dam와 HEAD NORMAL COORDINATE Y AXIS(Stage2) Collect Result_Dam의 상관관계: 0.997222491026016
Head Zero Position Y Collect Result_Dam와 Head Purge Position X Collect Result_Dam의 상관관계: -0.9991435701922116
Head Zero Position Z Collect Result_Dam와 Head Purge Position X Collect Result_Dam의 상관관계: -0.9991490400388723
Head Zero Position Z Collect Result_Dam와 Head Zero Position Y Collect Result_Dam의 상관관계: 0.9999168250383242
Machine Tact time Collect Result_Dam와 Head Purge Position X Collect Result_Dam의 상관관계: -0.9914490717908745
Machine Tact time Collect Result_Dam와 Head Zero Position Y Collect Result_Dam의 상관관계: 0.9928418093572455
Machine Tact time Collect Result_Dam와 Head Zero Position Z Collect Result_Dam의 상관

((40506, 78), (17361, 79))

### 범주형 변수 확인

In [10]:
# train_data의 변수 타입 데이터 프레임 생성
train_data_types = pd.DataFrame(train_data.dtypes, columns=["type"])
train_data_types['nunique'] = train_data.nunique()
train_data_types['unique'] = train_data.apply(lambda x: str(x.unique()), axis=0)
train_data_types['null_num'] = train_data.isnull().sum()
train_data_types['type'].value_counts()

train_data_types[train_data_types['type'] == 'object']

####  OK nan 만 있는 열 제거

train_data.drop(columns= ['HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Dam'], inplace=True)
test_data.drop(columns= ['HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Dam'], inplace=True)
train_data.shape, test_data.shape

le_cols = ['Equipment_Dam', 'Equipment_Fill1', 'Equipment_Fill2', 'Chamber Temp. Judge Value_AutoClave']
ohe_cols = ['Model.Suffix_Dam']
drop_cols = ['Workorder_Dam']

##### 라벨인코딩

from sklearn.preprocessing import LabelEncoder
import numpy as np

# 학습되지 않은 레이블을 새로운 값으로 처리
def encode_with_unseen(x, le):
    if x in le.classes_:
        return le.transform([x])[0]
    else:
        return len(le.classes_)  # unseen label에 대해 새로운 숫자 할당

for col in le_cols:
    le = LabelEncoder()

    # train_data와 test_data에 등장하는 모든 레이블을 미리 학습

    # train_data 변환
    train_data[col] = le.fit_transform(train_data[col])


    test_data[col] = test_data[col].apply(lambda x: encode_with_unseen(x, le))

train_data.shape, test_data.shape

# le_cols의 unique 값 확인
for col in le_cols:
    print(f"{col}: {train_data[col].unique()}")

##### 원핫인코딩

from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# 원핫인코더 생성
ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

for col in ohe_cols:
    # train_data에 대해 원핫인코딩 학습 및 변환
    ohe.fit(train_data[[col]])  # 2D로 입력해야 하므로 [[]] 사용
    train_encoded = ohe.transform(train_data[[col]])

    # test_data에 대해 동일한 인코더를 사용하여 변환
    test_encoded = ohe.transform(test_data[[col]])

    # 변환된 데이터프레임 생성 (원래 컬럼 이름을 기준으로 원핫 인코딩된 열의 이름을 생성)
    train_encoded_df = pd.DataFrame(train_encoded, columns=ohe.get_feature_names_out([col]))
    test_encoded_df = pd.DataFrame(test_encoded, columns=ohe.get_feature_names_out([col]))

    # 인코딩된 열을 원래 데이터프레임에 추가
    train_data = pd.concat([train_data, train_encoded_df], axis=1)
    test_data = pd.concat([test_data, test_encoded_df], axis=1)

    # 원래의 컬럼을 삭제 (원핫 인코딩된 데이터로 대체)
    train_data.drop(columns=[col], inplace=True)
    test_data.drop(columns=[col], inplace=True)

train_data.shape, test_data.shape

##### 컬럼 드랍

train_data.drop(columns=drop_cols, inplace=True)
test_data.drop(columns=drop_cols, inplace=True)
train_data.shape, test_data.shape

#### target label 변경

train_data.loc[train_data['target'] == 'Normal', 'target'] = 0
train_data.loc[train_data['target'] == 'AbNormal', 'target'] = 1
train_data['target'] = train_data['target'].astype(int)
train_data['target']

Equipment_Dam: [0 1]
Equipment_Fill1: [0 1]
Equipment_Fill2: [0 1]
Chamber Temp. Judge Value_AutoClave: [1 0]


0        0
1        0
2        0
3        0
4        0
        ..
40501    0
40502    0
40503    0
40504    0
40505    1
Name: target, Length: 40506, dtype: int32

#### OK가 오염된 열 확인 및 처리
['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam',

'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1',
       
'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2']

아에 제거 or 다른 컬럼과의 관계를 보고 대체


##### 다른 걸럼과의 관계를 보고 대체

In [ ]:
# 준형이가 해줄거임.. 아주 잘.. (기대하고 있겠음 ㅎ)

In [11]:
# prompt: ['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam',
# 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1',
# 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2']
# 이 열들의 OK값을 결측치로 바꿔줘

ok_leakage_cols = ['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam',
'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2']

for col in ok_leakage_cols:
  train_data[col] = train_data[col].replace('OK', np.nan)
  test_data[col] = test_data[col].replace('OK', np.nan)

In [12]:
# prompt: ['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2']을 float으로 바꿔줘

for col in ['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2']:
  train_data[col] = train_data[col].astype(float)
  test_data[col] = test_data[col].astype(float)

In [13]:
train_data['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam'].unique()

array([  nan, 550.3, 162.4, 549. , 549.5, 550. , 548.5])

In [14]:
import numpy as np

# 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam'의 결측치가 아닌 행들을 선택
not_null_data = train_data[train_data['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam'].notnull()]

# (X좌표 + Y좌표 + Z좌표)의 평균 계산
mean_value = (not_null_data['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam'] +
              not_null_data['HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Dam'] +
              not_null_data['HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Dam']).mean()

# 유니크 값 목록
unique_values = np.array([550.3, 162.4, 549, 549.5, 550, 548.5])

# 결측치를 채우는 함수 정의
def replace_with_closest_value(row, mean_value, unique_values):
    if np.isnan(row['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam']):
        # mean_value - Y - Z 계산
        calculated_value = mean_value - row['HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Dam'] - row['HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Dam']
        # 가장 가까운 유니크 값 선택
        closest_value = unique_values[np.abs(unique_values - calculated_value).argmin()]
        return closest_value
    else:
        return row['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam']

# train_data에서 결측치를 대체
train_data['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam'] = train_data.apply(
    lambda row: replace_with_closest_value(row, mean_value, unique_values), axis=1
)

# test_data에서도 동일한 방식 적용
test_data['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam'] = test_data.apply(
    lambda row: replace_with_closest_value(row, mean_value, unique_values), axis=1
)


In [15]:
train_data['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1'].unique()

array([  nan, 838.4, 837.7, 837.9, 838.2, 837.5])

In [16]:
import numpy as np

# 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam'의 결측치가 아닌 행들을 선택
not_null_data = train_data[train_data['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1'].notnull()]

# (X좌표 + Y좌표 + Z좌표)의 평균 계산
mean_value = (not_null_data['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1'] +
              not_null_data['HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Fill1'] +
              not_null_data['HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Fill1']).mean()

# 유니크 값 목록
unique_values = np.array([838.4, 837.7, 837.9, 838.2, 837.5])

# 결측치를 채우는 함수 정의
def replace_with_closest_value(row, mean_value, unique_values):
    if np.isnan(row['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1']):
        # mean_value - Y - Z 계산
        calculated_value = mean_value - row['HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Fill1'] - row['HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Fill1']
        # 가장 가까운 유니크 값 선택
        closest_value = unique_values[np.abs(unique_values - calculated_value).argmin()]
        return closest_value
    else:
        return row['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1']

# train_data에서 결측치를 대체
train_data['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1'] = train_data.apply(
    lambda row: replace_with_closest_value(row, mean_value, unique_values), axis=1
)

# test_data에서도 동일한 방식 적용
test_data['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1'] = test_data.apply(
    lambda row: replace_with_closest_value(row, mean_value, unique_values), axis=1
)


In [17]:
train_data['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2'].unique()

array([  nan, 835.5, 305. ])

In [18]:
import numpy as np

# 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam'의 결측치가 아닌 행들을 선택
not_null_data = train_data[train_data['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2'].notnull()]

# (X좌표 + Y좌표 + Z좌표)의 평균 계산
mean_value = (not_null_data['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2'] +
              not_null_data['HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Fill2'] +
              not_null_data['HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Fill2']).mean()

# 유니크 값 목록
unique_values = np.array([835.5, 305.0])

# 결측치를 채우는 함수 정의
def replace_with_closest_value(row, mean_value, unique_values):
    if np.isnan(row['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2']):
        # mean_value - Y - Z 계산
        calculated_value = mean_value - row['HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Fill2'] - row['HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Fill2']
        # 가장 가까운 유니크 값 선택
        closest_value = unique_values[np.abs(unique_values - calculated_value).argmin()]
        return closest_value
    else:
        return row['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2']

# train_data에서 결측치를 대체
train_data['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2'] = train_data.apply(
    lambda row: replace_with_closest_value(row, mean_value, unique_values), axis=1
)

# test_data에서도 동일한 방식 적용
test_data['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2'] = test_data.apply(
    lambda row: replace_with_closest_value(row, mean_value, unique_values), axis=1
)


In [19]:
# prompt: 데이터에 결측치가 있는지 확인해

# Check for missing values in train data
train_missing = train_data.isnull().sum()
print("Missing values in train data:\n", train_missing[train_missing > 0])

# Check for missing values in test data
test_missing = test_data.isnull().sum()
print("\nMissing values in test data:\n", test_missing[test_missing > 0])


Missing values in train data:
 Series([], dtype: int64)

Missing values in test data:
 target    17361
dtype: int64


##### 범주형 변수 잘 처리됐는지 확인

In [20]:
# train_data의 변수 타입 데이터 프레임 생성
train_data_types = pd.DataFrame(train_data.dtypes, columns=["type"])
train_data_types['nunique'] = train_data.nunique()
train_data_types['unique'] = train_data.apply(lambda x: str(x.unique()), axis=0)
train_data_types['null_num'] = train_data.isnull().sum()
train_data_types['type'].value_counts()

type
float64    51
int64      26
int32       5
Name: count, dtype: int64

In [21]:
train_data_types[train_data_types['type'] == 'object']

,type,nunique,unique,null_num


In [22]:
columns_list = train_data.columns.tolist()
columns = columns_list

# 공정별 4개의 데이터프레임으로 분할

In [23]:
# 접미사별 칼럼 구분
dam_columns = [col for col in columns if '_Dam' in col]
fill1_columns = [col for col in columns if '_Fill1' in col]
fill2_columns = [col for col in columns if '_Fill2' in col]
autoclave_columns = [col for col in columns if '_AutoClave' in col]
misc_columns = [col for col in columns if '_Dam' not in col and '_Fill1' not in col and '_Fill2' not in col and '_AutoClave' not in col]

# 데이터프레임 나누기
df_dam = train_data[dam_columns]
df_fill1 = train_data[fill1_columns]
df_fill2 = train_data[fill2_columns]
df_autoclave = train_data[autoclave_columns]
df_misc = train_data[misc_columns]

In [24]:
# target 칼럼을 각 데이터프레임에 추가하기 전에 target 칼럼이 존재하는지 확인
if 'target' in df_misc.columns:
    target_series = df_misc['target']
    
    # 각 데이터프레임에 target 칼럼 추가
    df_dam['target'] = target_series.values
    df_fill1['target'] = target_series.values
    df_fill2['target'] = target_series.values
    df_autoclave['target'] = target_series.values

    # 원래의 df_misc에서 target 칼럼 삭제
    df_misc.drop(columns=['target'], inplace=True)

In [25]:
# 기존의 columns 리스트
columns = [col for col in train_data.columns]

# 'Set ID'를 columns 리스트에 추가
columns.insert(0, 'Set ID')

# 접미사별 칼럼 구분
dam_columns = [col for col in columns if '_Dam' in col]
fill1_columns = [col for col in columns if '_Fill1' in col]
fill2_columns = [col for col in columns if '_Fill2' in col]
autoclave_columns = [col for col in columns if '_AutoClave' in col]
misc_columns = [col for col in columns if '_Dam' not in col and '_Fill1' not in col and '_Fill2' not in col and '_AutoClave' not in col]

# test_data에도 적용
df_dam_test = test_data[dam_columns]
df_fill1_test = test_data[fill1_columns]
df_fill2_test = test_data[fill2_columns]
df_autoclave_test = test_data[autoclave_columns]
df_misc_test = test_data[misc_columns]

df_dam_test = pd.concat([test_data[['Set ID']], df_dam_test], axis=1)
df_fill1_test = pd.concat([test_data[['Set ID']], df_fill1_test], axis=1)
df_fill2_test = pd.concat([test_data[['Set ID']], df_fill2_test], axis=1)
df_autoclave_test = pd.concat([test_data[['Set ID']], df_autoclave_test], axis=1)

In [26]:
# target 칼럼을 각 데이터프레임에 추가하기 전에 target 칼럼이 존재하는지 확인
if 'target' in df_misc_test.columns:
    target_series = df_misc_test['target']
    
    # 각 데이터프레임에 target 칼럼 추가
    df_dam_test['target'] = target_series.values
    df_fill1_test['target'] = target_series.values
    df_fill2_test['target'] = target_series.values
    df_autoclave_test['target'] = target_series.values

    # 원래의 df_misc에서 target 칼럼 삭제
    df_misc_test.drop(columns=['target'], inplace=True)

### 최종 예측

# 블렌딩

## df_dam

In [27]:
# stratify k fold 적용
from sklearn.model_selection import StratifiedKFold
from collections import Counter
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.combine import SMOTEENN, SMOTETomek
from collections import Counter

In [29]:
import xgboost as xgb
import lightgbm as lgb
import catboost as cat

rf = RandomForestClassifier(random_state=RANDOM_STATE)
xgb = xgb.XGBClassifier(random_state=RANDOM_STATE)
lgbm = lgb.LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)
cat = cat.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE)

In [30]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# train_data에서 target을 제외한 모든 열을 feature로 사용
X = df_dam.drop(columns=["target"])
y = df_dam["target"]

result_df = pd.DataFrame(columns=["fold", "sampling", "model", "f1", "precision", "recall", "accuracy"])

# train 데이터를 5개로 나누어 교차 검증
for i, index in enumerate(skf.split(X, y)):
    # 데이터 분할
    print(f"Fold {i + 1}")
    X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
    y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]

    train = pd.concat([X_train, y_train], axis=1)
    val = pd.concat([X_val, y_val], axis=1)

    X_basic, y_basic = X_train, y_train
    # 샘플링 파트
    print('샘플링 진행')
    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_ros, y_ros = ros.fit_resample(X_train, y_train)


    normal_ratio = 2.0
    df_normal = train[train["target"] == 0]
    df_abnormal = train[train["target"] == 1]

    df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
    df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
    X_under_2, y_under_2 = df_concat.drop(columns=["target"]), df_concat["target"]


    normal_ratio = 3.0
    df_normal = train[train["target"] == 0]
    df_abnormal = train[train["target"] == 1]

    df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
    df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
    X_under_3, y_under_3 = df_concat.drop(columns=["target"]), df_concat["target"]






    for sample_tech in tqdm(["basic", "ros", "under_2", "under_3"]):
        print(sample_tech, '데이터 학습 진행')
        train_x = globals()[f"X_{sample_tech}"]
        train_y = globals()[f"y_{sample_tech}"]
        for model in [rf, xgb, lgbm, cat]:
            print(model.__class__.__name__)
            model.fit(train_x, train_y)
            y_pred = model.predict(X_val)
            f1 = f1_score(y_val, y_pred)
            precision = precision_score(y_val, y_pred)
            recall = recall_score(y_val, y_pred)
            accuracy = accuracy_score(y_val, y_pred)
            result_df.loc[len(result_df), :] = [i+1, sample_tech, model.__class__.__name__, f1, precision, recall, accuracy]
        del train_x, train_y

Fold 1
샘플링 진행
Total: Normal: 30524, AbNormal: 1880
Total: Normal: 30524, AbNormal: 1880


  0%|          | 0/4 [00:00<?, ?it/s]

basic 데이터 학습 진행
RandomForestClassifier
XGBClassifier
LGBMClassifier
CatBoostClassifier


 25%|██▌       | 1/4 [00:18<00:54, 18.21s/it]

ros 데이터 학습 진행
RandomForestClassifier
XGBClassifier
LGBMClassifier
CatBoostClassifier


...

 50%|█████     | 2/4 [01:23<01:25, 42.86s/it]

under_2 데이터 학습 진행
RandomForestClassifier
XGBClassifier
LGBMClassifier
CatBoostClassifier


 75%|███████▌  | 3/4 [01:33<00:27, 27.87s/it]

under_3 데이터 학습 진행
RandomForestClassifier
XGBClassifier
LGBMClassifier
CatBoostClassifier


100%|██████████| 4/4 [01:44<00:00, 26.09s/it]


In [32]:
result_df.groupby(["sampling", "model"]).mean()

fold        f1 precision    recall  accuracy
sampling model                                                              
basic    CatBoostClassifier      3.0  0.069897  0.786768  0.036596  0.943539
         LGBMClassifier          3.0   0.04799  0.903137  0.024681  0.943243
         RandomForestClassifier  3.0  0.095436  0.256216  0.058723  0.935466
         XGBClassifier           3.0  0.055461  0.668139  0.028936  0.942823
ros      CatBoostClassifier      3.0  0.156459  0.099016  0.372766    0.7668
         LGBMClassifier          3.0  0.160157  0.095921  0.485532   0.70439
         RandomForestClassifier  3.0  0.119958  0.163291  0.094894  0.919395
         XGBClassifier           3.0  0.154741  0.096532  0.390213  0.752407
under_2  CatBoostClassifier      3.0  0.153738  0.132278   0.18383  0.882635
         LGBMClassifier          3.0  0.153879  0.114782  0.233617  0.850911
         RandomForestClassifier  3.0  0.153618  0.099955  0.331915  0.787785
         XGBClassifier           3.0   0.14675  0.099068  0.283404  0.808769
under_3  CatBoostClassifier      3.0  0.110218  0.228436  0.073191  0.931245
         LGBMClassifier          3.0   0.12893  0.168265  0.105532  0.917494
         RandomForestClassifier  3.0  0.151459   0.11497  0.222128  0.855602
         XGBClassifier           3.0   0.15484  0.140525  0.173191  0.890535

In [33]:
# stratify k fold 적용
from sklearn.model_selection import StratifiedKFold
from collections import Counter
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.combine import SMOTEENN, SMOTETomek
from collections import Counter
import xgboost as xgb
import lightgbm as lgb
import catboost as cat

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# train_data에서 target을 제외한 모든 열을 feature로 사용
X = df_dam.drop(columns=["target"])
y = df_dam["target"]

blending_result_df = pd.DataFrame(columns=["fold", "f1", "precision", "recall", "accuracy"])
y_proba_list = []
# train 데이터를 5개로 나누어 교차 검증
for i, index in enumerate(skf.split(X, y)):
    # 데이터 분할
    print(f"Fold {i + 1}")
    X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
    y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
    print(Counter(y_train), Counter(y_val))

    train = pd.concat([X_train, y_train], axis=1)
    val = pd.concat([X_val, y_val], axis=1)

    # 샘플링 파트
    print('샘플링 진행')
    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_ros, y_ros = ros.fit_resample(X_train, y_train)

    normal_ratio = 2.0
    df_normal = train[train["target"] == 0]
    df_abnormal = train[train["target"] == 1]
    print(f"Total: Normal: {len(df_normal)}, AbNormal: {len(df_abnormal)}")

    df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
    df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
    X_under_2, y_under_2 = df_concat.drop(columns=["target"]), df_concat["target"]

    print("언더샘플링 후 train의 클래스 비율:", Counter(y_under_2))

    normal_ratio = 3.0
    df_normal = train[train["target"] == 0]
    df_abnormal = train[train["target"] == 1]
    print(f"Total: Normal: {len(df_normal)}, AbNormal: {len(df_abnormal)}")

    df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
    df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
    X_under_3, y_under_3 = df_concat.drop(columns=["target"]), df_concat["target"]

    print("언더샘플링 후 train의 클래스 비율:", Counter(y_under_3))

    print('모델 학습 진행')
    import catboost as cat
    ros_cat = cat.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE)
    under2_cat = cat.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE)
    import xgboost as xgb
    under3_rf = RandomForestClassifier(random_state=RANDOM_STATE)


    ros_cat.fit(X_ros,y_ros)
    under2_cat.fit(X_under_2, y_under_2)
    under3_rf.fit(X_under_3, y_under_3)

    ros_cat_pred = ros_cat.predict_proba(X_val)[:, 1]
    under2_cat_pred = under2_cat.predict_proba(X_val)[:, 1]
    under3_rf_pred = under3_rf.predict_proba(X_val)[:, 1]

    final_outputs = {
        'ros_cat' : ros_cat_pred,
        'under2_cat' : under2_cat_pred,
        'under3_rf' : under3_rf_pred}

    #Blending
    y_pred = final_outputs['ros_cat'] * 0.5 +final_outputs['under2_cat'] * 0.3 +final_outputs['under3_rf'] * 0.2
    y_proba_list.append(final_outputs['ros_cat'] * 0.5 +final_outputs['under2_cat'] * 0.3 +final_outputs['under3_rf'] * 0.2)
    y_pred = np.where(y_pred > 0.5, 1, 0)

    f1 = f1_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    accuracy = accuracy_score(y_val, y_pred)
    blending_result_df.loc[len(blending_result_df), :] = [i+1, f1, precision, recall, accuracy]

len(y_val), len(ros_cat_pred)

y_proba_list[3].shape

def calculate_f1_threshold(threshold):
    y_pred = (y_proba >= threshold).astype(int)
    return f1_score(y_val, y_pred)

f1_scores_list = []
for i, index in enumerate(skf.split(X, y)):
    # 데이터 분할
    print(f"Fold {i + 1}")
    X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
    y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
    y_proba = y_proba_list[i]
    # F1 스코어 계산
    thresholds = np.arange(0.4, 0.6, 0.0001)

    # 모든 임계값에 대해 F1 스코어 계산
    f1_scores = [calculate_f1_threshold(t) for t in thresholds]
    f1_scores_list.append(f1_scores)

pd.DataFrame(f1_scores_list)

pd.DataFrame(f1_scores_list).mean(axis=0).idxmax()

np.arange(0.4, 0.6, 0.0001)[1110]

pd.DataFrame(f1_scores_list)[1110].mean()

blending_result_df

blending_result_df['f1'].mean()

Fold 1
Counter({0: 30524, 1: 1880}) Counter({0: 7632, 1: 470})
샘플링 진행
Total: Normal: 30524, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 3760, 1: 1880})
Total: Normal: 30524, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 5640, 1: 1880})
모델 학습 진행
Fold 2
Counter({0: 30525, 1: 1880}) Counter({0: 7631, 1: 470})
샘플링 진행
Total: Normal: 30525, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 3760, 1: 1880})
Total: Normal: 30525, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 5640, 1: 1880})
모델 학습 진행
Fold 3
Counter({0: 30525, 1: 1880}) Counter({0: 7631, 1: 470})
샘플링 진행
Total: Normal: 30525, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 3760, 1: 1880})
Total: Normal: 30525, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 5640, 1: 1880})
모델 학습 진행
Fold 4
Counter({0: 30525, 1: 1880}) Counter({0: 7631, 1: 470})
샘플링 진행
Total: Normal: 30525, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 3760, 1: 1880})
Total: Normal: 30525, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 5640,

0.16620514817745

## df_fill1

In [34]:
# stratify k fold 적용
from sklearn.model_selection import StratifiedKFold
from collections import Counter
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.combine import SMOTEENN, SMOTETomek
from collections import Counter
import xgboost as xgb
import lightgbm as lgb
import catboost as cat

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# train_data에서 target을 제외한 모든 열을 feature로 사용
X = df_fill1.drop(columns=["target"])
y = df_fill1["target"]

blending_result_df = pd.DataFrame(columns=["fold", "f1", "precision", "recall", "accuracy"])
y_proba_list = []
# train 데이터를 5개로 나누어 교차 검증
for i, index in enumerate(skf.split(X, y)):
    # 데이터 분할
    print(f"Fold {i + 1}")
    X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
    y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
    print(Counter(y_train), Counter(y_val))

    train = pd.concat([X_train, y_train], axis=1)
    val = pd.concat([X_val, y_val], axis=1)

    # 샘플링 파트
    print('샘플링 진행')
    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_ros, y_ros = ros.fit_resample(X_train, y_train)

    normal_ratio = 2.0
    df_normal = train[train["target"] == 0]
    df_abnormal = train[train["target"] == 1]
    print(f"Total: Normal: {len(df_normal)}, AbNormal: {len(df_abnormal)}")

    df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
    df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
    X_under_2, y_under_2 = df_concat.drop(columns=["target"]), df_concat["target"]

    print("언더샘플링 후 train의 클래스 비율:", Counter(y_under_2))

    normal_ratio = 3.0
    df_normal = train[train["target"] == 0]
    df_abnormal = train[train["target"] == 1]
    print(f"Total: Normal: {len(df_normal)}, AbNormal: {len(df_abnormal)}")

    df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
    df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
    X_under_3, y_under_3 = df_concat.drop(columns=["target"]), df_concat["target"]

    print("언더샘플링 후 train의 클래스 비율:", Counter(y_under_3))

    print('모델 학습 진행')
    import catboost as cat
    ros_cat = cat.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE)
    under2_cat = cat.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE)
    import xgboost as xgb
    under3_rf = RandomForestClassifier(random_state=RANDOM_STATE)


    ros_cat.fit(X_ros,y_ros)
    under2_cat.fit(X_under_2, y_under_2)
    under3_rf.fit(X_under_3, y_under_3)

    ros_cat_pred = ros_cat.predict_proba(X_val)[:, 1]
    under2_cat_pred = under2_cat.predict_proba(X_val)[:, 1]
    under3_rf_pred = under3_rf.predict_proba(X_val)[:, 1]

    final_outputs = {
        'ros_cat' : ros_cat_pred,
        'under2_cat' : under2_cat_pred,
        'under3_rf' : under3_rf_pred}

    #Blending
    y_pred = final_outputs['ros_cat'] * 0.5 +final_outputs['under2_cat'] * 0.3 +final_outputs['under3_rf'] * 0.2
    y_proba_list.append(final_outputs['ros_cat'] * 0.5 +final_outputs['under2_cat'] * 0.3 +final_outputs['under3_rf'] * 0.2)
    y_pred = np.where(y_pred > 0.5, 1, 0)

    f1 = f1_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    accuracy = accuracy_score(y_val, y_pred)
    blending_result_df.loc[len(blending_result_df), :] = [i+1, f1, precision, recall, accuracy]

len(y_val), len(ros_cat_pred)

y_proba_list[3].shape

def calculate_f1_threshold(threshold):
    y_pred = (y_proba >= threshold).astype(int)
    return f1_score(y_val, y_pred)

f1_scores_list = []
for i, index in enumerate(skf.split(X, y)):
    # 데이터 분할
    print(f"Fold {i + 1}")
    X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
    y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
    y_proba = y_proba_list[i]
    # F1 스코어 계산
    thresholds = np.arange(0.4, 0.6, 0.0001)

    # 모든 임계값에 대해 F1 스코어 계산
    f1_scores = [calculate_f1_threshold(t) for t in thresholds]
    f1_scores_list.append(f1_scores)

pd.DataFrame(f1_scores_list)

pd.DataFrame(f1_scores_list).mean(axis=0).idxmax()

np.arange(0.4, 0.6, 0.0001)[1110]

pd.DataFrame(f1_scores_list)[1110].mean()

blending_result_df

blending_result_df['f1'].mean()

Fold 1
Counter({0: 30524, 1: 1880}) Counter({0: 7632, 1: 470})
샘플링 진행
Total: Normal: 30524, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 3760, 1: 1880})
Total: Normal: 30524, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 5640, 1: 1880})
모델 학습 진행
Fold 2
Counter({0: 30525, 1: 1880}) Counter({0: 7631, 1: 470})
샘플링 진행
Total: Normal: 30525, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 3760, 1: 1880})
Total: Normal: 30525, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 5640, 1: 1880})
모델 학습 진행
Fold 3
Counter({0: 30525, 1: 1880}) Counter({0: 7631, 1: 470})
샘플링 진행
Total: Normal: 30525, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 3760, 1: 1880})
Total: Normal: 30525, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 5640, 1: 1880})
모델 학습 진행
Fold 4
Counter({0: 30525, 1: 1880}) Counter({0: 7631, 1: 470})
샘플링 진행
Total: Normal: 30525, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 3760, 1: 1880})
Total: Normal: 30525, AbNormal: 1880
언더샘플링 후 train의 클래스 비율: Counter({0: 5640,

0.15319579386922003

## df_fill2

In [ ]:
# stratify k fold 적용
from sklearn.model_selection import StratifiedKFold
from collections import Counter
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.combine import SMOTEENN, SMOTETomek
from collections import Counter
import xgboost as xgb
import lightgbm as lgb
import catboost as cat

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# train_data에서 target을 제외한 모든 열을 feature로 사용
X = df_fill2.drop(columns=["target"])
y = df_fill2["target"]

blending_result_df = pd.DataFrame(columns=["fold", "f1", "precision", "recall", "accuracy"])
y_proba_list = []
# train 데이터를 5개로 나누어 교차 검증
for i, index in enumerate(skf.split(X, y)):
    # 데이터 분할
    print(f"Fold {i + 1}")
    X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
    y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
    print(Counter(y_train), Counter(y_val))

    train = pd.concat([X_train, y_train], axis=1)
    val = pd.concat([X_val, y_val], axis=1)

    # 샘플링 파트
    print('샘플링 진행')
    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_ros, y_ros = ros.fit_resample(X_train, y_train)

    normal_ratio = 2.0
    df_normal = train[train["target"] == 0]
    df_abnormal = train[train["target"] == 1]
    print(f"Total: Normal: {len(df_normal)}, AbNormal: {len(df_abnormal)}")

    df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
    df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
    X_under_2, y_under_2 = df_concat.drop(columns=["target"]), df_concat["target"]

    print("언더샘플링 후 train의 클래스 비율:", Counter(y_under_2))

    normal_ratio = 3.0
    df_normal = train[train["target"] == 0]
    df_abnormal = train[train["target"] == 1]
    print(f"Total: Normal: {len(df_normal)}, AbNormal: {len(df_abnormal)}")

    df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
    df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
    X_under_3, y_under_3 = df_concat.drop(columns=["target"]), df_concat["target"]

    print("언더샘플링 후 train의 클래스 비율:", Counter(y_under_3))

    print('모델 학습 진행')
    import catboost as cat
    ros_cat = cat.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE)
    under2_cat = cat.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE)
    import xgboost as xgb
    under3_rf = RandomForestClassifier(random_state=RANDOM_STATE)


    ros_cat.fit(X_ros,y_ros)
    under2_cat.fit(X_under_2, y_under_2)
    under3_rf.fit(X_under_3, y_under_3)

    ros_cat_pred = ros_cat.predict_proba(X_val)[:, 1]
    under2_cat_pred = under2_cat.predict_proba(X_val)[:, 1]
    under3_rf_pred = under3_rf.predict_proba(X_val)[:, 1]

    final_outputs = {
        'ros_cat' : ros_cat_pred,
        'under2_cat' : under2_cat_pred,
        'under3_rf' : under3_rf_pred}

    #Blending
    y_pred = final_outputs['ros_cat'] * 0.5 +final_outputs['under2_cat'] * 0.3 +final_outputs['under3_rf'] * 0.2
    y_proba_list.append(final_outputs['ros_cat'] * 0.5 +final_outputs['under2_cat'] * 0.3 +final_outputs['under3_rf'] * 0.2)
    y_pred = np.where(y_pred > 0.5, 1, 0)

    f1 = f1_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    accuracy = accuracy_score(y_val, y_pred)
    blending_result_df.loc[len(blending_result_df), :] = [i+1, f1, precision, recall, accuracy]

len(y_val), len(ros_cat_pred)

y_proba_list[3].shape

def calculate_f1_threshold(threshold):
    y_pred = (y_proba >= threshold).astype(int)
    return f1_score(y_val, y_pred)

f1_scores_list = []
for i, index in enumerate(skf.split(X, y)):
    # 데이터 분할
    print(f"Fold {i + 1}")
    X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
    y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
    y_proba = y_proba_list[i]
    # F1 스코어 계산
    thresholds = np.arange(0.4, 0.6, 0.0001)

    # 모든 임계값에 대해 F1 스코어 계산
    f1_scores = [calculate_f1_threshold(t) for t in thresholds]
    f1_scores_list.append(f1_scores)

pd.DataFrame(f1_scores_list)

pd.DataFrame(f1_scores_list).mean(axis=0).idxmax()

np.arange(0.4, 0.6, 0.0001)[1110]

pd.DataFrame(f1_scores_list)[1110].mean()

blending_result_df

blending_result_df['f1'].mean()

## df_autoclave

In [ ]:
# stratify k fold 적용
from sklearn.model_selection import StratifiedKFold
from collections import Counter
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.combine import SMOTEENN, SMOTETomek
from collections import Counter
import xgboost as xgb
import lightgbm as lgb
import catboost as cat

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# train_data에서 target을 제외한 모든 열을 feature로 사용
X = df_autoclave.drop(columns=["target"])
y = df_autoclave["target"]

blending_result_df = pd.DataFrame(columns=["fold", "f1", "precision", "recall", "accuracy"])
y_proba_list = []
# train 데이터를 5개로 나누어 교차 검증
for i, index in enumerate(skf.split(X, y)):
    # 데이터 분할
    print(f"Fold {i + 1}")
    X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
    y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
    print(Counter(y_train), Counter(y_val))

    train = pd.concat([X_train, y_train], axis=1)
    val = pd.concat([X_val, y_val], axis=1)

    # 샘플링 파트
    print('샘플링 진행')
    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_ros, y_ros = ros.fit_resample(X_train, y_train)

    normal_ratio = 2.0
    df_normal = train[train["target"] == 0]
    df_abnormal = train[train["target"] == 1]
    print(f"Total: Normal: {len(df_normal)}, AbNormal: {len(df_abnormal)}")

    df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
    df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
    X_under_2, y_under_2 = df_concat.drop(columns=["target"]), df_concat["target"]

    print("언더샘플링 후 train의 클래스 비율:", Counter(y_under_2))

    normal_ratio = 3.0
    df_normal = train[train["target"] == 0]
    df_abnormal = train[train["target"] == 1]
    print(f"Total: Normal: {len(df_normal)}, AbNormal: {len(df_abnormal)}")

    df_normal = df_normal.sample(n=int(len(df_abnormal) * normal_ratio), replace=False, random_state=RANDOM_STATE)
    df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
    X_under_3, y_under_3 = df_concat.drop(columns=["target"]), df_concat["target"]

    print("언더샘플링 후 train의 클래스 비율:", Counter(y_under_3))

    print('모델 학습 진행')
    import catboost as cat
    ros_cat = cat.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE)
    under2_cat = cat.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE)
    import xgboost as xgb
    under3_rf = RandomForestClassifier(random_state=RANDOM_STATE)


    ros_cat.fit(X_ros,y_ros)
    under2_cat.fit(X_under_2, y_under_2)
    under3_rf.fit(X_under_3, y_under_3)

    ros_cat_pred = ros_cat.predict_proba(X_val)[:, 1]
    under2_cat_pred = under2_cat.predict_proba(X_val)[:, 1]
    under3_rf_pred = under3_rf.predict_proba(X_val)[:, 1]

    final_outputs = {
        'ros_cat' : ros_cat_pred,
        'under2_cat' : under2_cat_pred,
        'under3_rf' : under3_rf_pred}

    #Blending
    y_pred = final_outputs['ros_cat'] * 0.5 +final_outputs['under2_cat'] * 0.3 +final_outputs['under3_rf'] * 0.2
    y_proba_list.append(final_outputs['ros_cat'] * 0.5 +final_outputs['under2_cat'] * 0.3 +final_outputs['under3_rf'] * 0.2)
    y_pred = np.where(y_pred > 0.5, 1, 0)

    f1 = f1_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    accuracy = accuracy_score(y_val, y_pred)
    blending_result_df.loc[len(blending_result_df), :] = [i+1, f1, precision, recall, accuracy]

len(y_val), len(ros_cat_pred)

y_proba_list[3].shape

def calculate_f1_threshold(threshold):
    y_pred = (y_proba >= threshold).astype(int)
    return f1_score(y_val, y_pred)

f1_scores_list = []
for i, index in enumerate(skf.split(X, y)):
    # 데이터 분할
    print(f"Fold {i + 1}")
    X_train, X_val = X.iloc[index[0]], X.iloc[index[1]]
    y_train, y_val = y.iloc[index[0]], y.iloc[index[1]]
    y_proba = y_proba_list[i]
    # F1 스코어 계산
    thresholds = np.arange(0.4, 0.6, 0.0001)

    # 모든 임계값에 대해 F1 스코어 계산
    f1_scores = [calculate_f1_threshold(t) for t in thresholds]
    f1_scores_list.append(f1_scores)

pd.DataFrame(f1_scores_list)

pd.DataFrame(f1_scores_list).mean(axis=0).idxmax()

np.arange(0.4, 0.6, 0.0001)[1110]

pd.DataFrame(f1_scores_list)[1110].mean()

blending_result_df

blending_result_df['f1'].mean()

## optuna를 사용한 하이퍼 파라미터 튜닝

#### rf

In [ ]:
import optuna

# Objective 함수 정의
def objective(trial):
    # 하이퍼파라미터 검색 공간 정의
    n_estimators = trial.suggest_int('n_estimators', 50, 150)
    max_depth = trial.suggest_int('max_depth', 5, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    max_features = trial.suggest_categorical('max_features', ['auto', 'sqrt', 'log2', 0.1, 0.5])
    # class_weight 하이퍼파라미터를 조정
    class_weight = trial.suggest_categorical('class_weight', ['balanced', None])

    # RandomForest 모델 생성
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight=class_weight,
        random_state=RANDOM_STATE
    )

    # 모델 훈련
    model.fit(X_under, y_under)

    # 예측 및 정확도 계산
    y_pred = model.predict(val_x)
    f1 = f1_score(val_y, y_pred, pos_label=1)

    return f1

# 3. Optuna 스터디 생성 및 최적화
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)  # 50번의 trial을 통해 최적화 수행

# 4. 최적 하이퍼파라미터 및 최적 성과 출력
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")


### cat

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


# Optuna의 Objective 함수 정의
def objective(trial):
    # 하이퍼파라미터 정의
    param = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 4, 12),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
        'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 1e-3, 1e+3),
        'border_count': trial.suggest_int('border_count', 5, 255),
        'scale_pos_weight': trial.suggest_loguniform('scale_pos_weight', 1e-3, 1e+3),
        'thread_count': -1  # 자동으로 사용할 스레드 수 결정
    }

    model = CatBoostClassifier(**param, verbose=0, random_state=RANDOM_STATE)
    model.fit(X_under, y_under)

    # Validation accuracy
    y_pred = model.predict(val_x)
    f1 = f1_score(val_x, y_pred)

    return f1

# Optuna Study 생성
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# 최적 하이퍼파라미터 출력
print("Best parameters:", study.best_params)
print("Best accuracy:", study.best_value)


### lgbm

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


# Optuna의 Objective 함수 정의
def objective(trial):
    # 하이퍼파라미터 정의
    param = {
        'objective': 'binary',
        'metric': 'binary_error',
        'boosting_type': trial.suggest_categorical('boosting_type', ['gbdt', 'dart', 'goss']),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'max_depth': trial.suggest_int('max_depth', -1, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'min_child_samples': trial.suggest_int('min_child_samples', 1, 100),
        'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-4, 1e+1),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-4, 1e+1),
    }

    model = lgb.LGBMClassifier(**param, random_state=RANDOM_STATE)

    model.fit(X_under, y_under)

    # Validation accuracy
    y_pred = model.predict(val_x)
    f1 = f1_score(val_x, y_pred)

    return f1

# Optuna Study 생성
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# 최적 하이퍼파라미터 출력
print("Best parameters:", study.best_params)
print("Best accuracy:", study.best_value)


### xgb

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


# Optuna의 Objective 함수 정의
def objective(trial):
    # 하이퍼파라미터 정의
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'booster': trial.suggest_categorical('booster', ['gbtree', 'dart', 'gblinear']),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 100),
        'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-4, 1e+1),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-4, 1e+1),
        'scale_pos_weight': trial.suggest_loguniform('scale_pos_weight', 1e-3, 1e+3)
    }

    model = xgb.XGBClassifier(**param, random_state=RANDOM_STATE)

    model.fit(X_under, y_under)

    # Validation accuracy
    y_pred = model.predict(val_x)
    f1 = f1_score(val_x, y_pred)

    return f1

# Optuna Study 생성
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# 최적 하이퍼파라미터 출력
print("Best parameters:", study.best_params)
print("Best accuracy:", study.best_value)


### 테스트 데이터 예측


In [ ]:
best_params = study.best_params
best_model = RandomForestClassifier(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    min_samples_leaf=best_params['min_samples_leaf'],
    max_features=best_params['max_features'],
    class_weight=best_params['class_weight'],
    random_state=RANDOM_STATE
)

In [ ]:
best_model.fit(X_under, y_under)

In [ ]:
best_model

In [ ]:
# final_df = pd.concat([df_concat, f_ab_df], axis=0).reset_index(drop=True)

# f_train_x = final_df[features]
# f_train_y = final_df["target"]

# model.fit(f_train_x, f_train_y)

In [ ]:
cat.fit(X_under, y_under)

In [ ]:
df_test_x = test_data.drop(columns=["Set ID", 'target'])

In [ ]:
test_pred = cat.predict(df_test_x)
test_pred

### 제출 파일 작성
파일 제출 전에 이름, 컬럼명, ID 있는지 잘 확인후 제출!

In [ ]:
# 제출 데이터 읽어오기 (df_test는 전처리된 데이터가 저장됨)
df_sub = pd.read_csv(os.path.join(ROOT_DIR, "submission.csv"))
df_sub["target"] = test_pred
df_sub.loc[df_sub["target"] == 0, 'target'] = "Normal"
df_sub.loc[df_sub["target"] == 1, 'target'] = "AbNormal"
# 제출 파일 저장
df_sub.to_csv("전처리추가_catboost.csv", index=False)

## SHAP

In [ ]:
import shap

# XGBoost 분류기 모델 초기화
cat.fit(X_under, y_under)


# SHAP 값 계산
explainer = shap.TreeExplainer(cat)
shap_values = explainer.shap_values(val_x)

# SHAP 값의 평균 절대값 계산
shap_values_df = pd.DataFrame(shap_values, columns=val_x.columns)
shap_importance = shap_values_df.abs().mean().sort_values(ascending=False)
shap_importance

In [ ]:
# shap_importance 0.1 이상인 컬럼만 선택
selected_cols = shap_importance[shap_importance > 0.015].index
len(selected_cols)

In [ ]:
shap_importance[shap_importance > 0.015].sum()

In [ ]:
# 상위 20개 주요 변수 보여줌
shap.summary_plot(shap_values, val_x, max_display=val_x.shape[1])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 10))  # 너비 14인치, 높이 10인치로 설정

# SHAP dependence plot 생성
shap.dependence_plot(80, shap_values, val_x, ax=ax)

# 그래프 표시
plt.show()

In [ ]:
shap.initjs()
sample_index = 0
shap.force_plot(explainer.expected_value, shap_values[sample_index], val_x.iloc[sample_index, :])